In [ ]:
!pip install requests beautifulsoup4 openai langchain-openai

In [ ]:
import requests
from bs4 import BeautifulSoup

def extract_text_from_url(url):
  response = requests.get(url)

  if response.status_code != 200:
    raise Exception(f'Failed to fetch URL {url}')
    return None

  else:
    soup = BeautifulSoup(response.text, 'html.parser')

    # Remove scripts e estilos
    for script_or_style in soup(['script', 'style']):
      script_or_style.extract()

    # Extrai texto bruto
    text = soup.get_text(separator=' ', strip=True)

    # Remove múltiplos espaços e quebras de linha
    lines = (line.strip() for line in text.splitlines())
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    text = ' '.join(chunk for chunk in chunks if chunk)

  text = soup.get_text()
  return text

# Exemplo de uso
resultado = extract_text_from_url('https://dev.to/kenakamu/azure-open-ai-in-vnet-3alo')
print(resultado)

In [ ]:
from langchain_openai.chat_models.azure import AzureChatOpenAI

client = AzureChatOpenAI(
    azure_endpoint = "https://azure-openai-dio-bootcamp-001.openai.azure.com",
    api_key = "1Talwmb3Z2yo80uLjEtOa8w5wndJ5Fe5Yw7PXFNHHs7q1xzpFDraJQQJ99BJACHYHv6XJ3w3AAABACOGDjEL",
    api_version="2024-02-15-preview",
    deployment_name= "gpt-4o-mini",
    max_retries=0
)

def translate_article(text, lang):
  messages = [
      ("system", "Você atua como tradutor de textos"),
      ("user", f"Traduza o {text} para o idioma {lang} e responda em markdown")
  ]

  response = client.invoke(messages)
  print(response.content)
  return response.content

translate_article("Let's see if the deployment was succeeded.", "português")

In [ ]:
url = 'https://dev.to/kenakamu/azure-open-ai-in-vnet-3alo'
text = extract_text_from_url(url)
article = translate_article(text, "pt-br")
print(article)